# Example 06d: self-contained up-down machine symmetrisation

This notebook demonstrates how to turn an **approximately** up-down symmetric
machine description into the exactly symmetric description required by reduced
even-in-$Z$ calculations. It uses a compact MAST-U-like arrangement defined
entirely below: no UDA access, pickle files, or external machine data are
required.

The notebook workflow has three distinct jobs:

1. identify reflected upper/lower elements and electrically odd circuits;
2. use preparation diagnostics and an independent magnetic audit to quantify
   the proposed change before the user accepts it;
3. return an exactly symmetric description and current-coordinate transforms.

For the reflection $\mathcal{R}_Z:(R,Z)\mapsto(R,2Z_0-Z)$, paired geometry is
replaced by

$$
\mathbf{x}_{\mathrm{sym}}=\frac{1}{2}\left(\mathbf{x}_{u}+
\mathcal{R}_Z\mathbf{x}_{l}\right).
$$

Currents in an independently driven upper/lower pair are decomposed as

$$
I_+=\frac{I_u+I_l}{2},\qquad I_-=\frac{I_u-I_l}{2}.
$$

Only $I_+$ belongs to the strict symmetric machine. The original currents are
still recoverable when both coordinates are retained:
$I_u=I_++I_-$ and $I_l=I_+-I_-$.


## 1. Define an imperfect MAST-U-like source machine

The source intentionally contains:

- a central solenoid;
- independently described upper/lower shaping and divertor coils;
- paired passive wall elements;
- a small common vertical offset and millimetre-scale pair mismatches;
- one anti-series vertical-control circuit, which is electrically odd.

The preparation operates on deep copies, so the source dictionaries are never
modified. The returned candidate is exactly symmetric and retains the
pre-averaging discrepancies needed for the subsequent acceptance checks.


In [ ]:
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np

from freegsnke import build_machine, equilibrium_update
from freegsnke.up_down_symmetry import prepare_up_down_symmetric_machine

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})


def filament(r, z, *, multiplier=80, polarity=1):
    """Return one rectangular active-coil filament bundle."""
    return {
        "R": [r],
        "Z": [z],
        "dR": 0.07,
        "dZ": 0.07,
        "resistivity": 1.55e-8,
        "polarity": polarity,
        "multiplier": multiplier,
    }


def rectangle(name, group, r, z, half_width=0.025):
    """Return one quadrilateral passive element."""
    return {
        "name": name,
        "element": group,
        "R": np.array(
            [r - half_width, r + half_width, r + half_width, r - half_width]
        ),
        "Z": np.array(
            [z - half_width, z - half_width, z + half_width, z + half_width]
        ),
        "resistivity": 5.5e-7,
    }


def mastu_like_source_machine():
    """Build an in-memory, approximately symmetric MAST-U-like description."""
    z0 = 0.008  # unknown to the preparation: an 8 mm global source offset
    active = {
        "solenoid": {
            **filament(0.24, z0, multiplier=20),
            "R": [0.24] * 6,
            "Z": z0 + np.array([-1.05, -0.63, -0.21, 0.21, 0.63, 1.05]),
        }
    }

    # Independent upper/lower circuits with small source-description errors.
    locations = {
        "P4": (0.43, 1.18),
        "P5": (0.66, 1.34),
        "D1": (1.35, 1.20),
        "D2": (1.63, 0.87),
        "DP": (1.72, 0.48),
        "PX": (1.76, 0.17),
    }
    for index, (name, (r, z)) in enumerate(locations.items()):
        dr = (index - 2.5) * 0.0007
        dz = (-1) ** index * 0.002
        active[f"{name}_upper"] = filament(r, z0 + z)
        active[f"{name}_lower"] = filament(r + dr, z0 - z + dz)

    # Opposite winding polarities make this circuit odd under reflection.
    active["vertical_control"] = {
        "upper": filament(1.83, z0 + 0.31),
        "lower": filament(1.83, z0 - 0.31, polarity=-1),
    }

    passive = []
    angles = np.linspace(np.pi / 18, 17 * np.pi / 18, 9)
    for index, angle in enumerate(angles):
        r = 1.02 + 0.63 * np.cos(angle)
        z = 1.02 * np.sin(angle)
        dr = 0.0015 * np.sin(2 * angle)
        dz = 0.0015 * np.cos(angle)
        passive.extend(
            [
                rectangle(f"wall_{index}_upper", f"wall_{index}", r, z0 + z),
                rectangle(
                    f"wall_{index}_lower",
                    f"wall_{index}",
                    r + dr,
                    z0 - z + dz,
                ),
            ]
        )

    theta = np.linspace(0.0, 2 * np.pi, 120, endpoint=False)
    r_limiter = 0.98 + 0.47 * np.cos(theta + 0.13 * np.sin(theta))
    z_limiter = z0 + 0.90 * np.sin(theta)
    limiter = [
        {"R": float(r), "Z": float(z)} for r, z in zip(r_limiter, z_limiter)
    ]

    r_wall = 0.98 + 0.54 * np.cos(theta + 0.11 * np.sin(theta))
    z_wall = z0 + 1.08 * np.sin(theta)
    wall = [{"R": float(r), "Z": float(z)} for r, z in zip(r_wall, z_wall)]
    return active, passive, limiter, wall


def active_parts(circuit):
    """Yield direct and compound active-coil parts in one form."""
    if "R" in circuit:
        return [circuit]
    return [
        part
        for part in circuit.values()
        if isinstance(part, dict) and "R" in part
    ]


def centroid(element):
    return float(np.mean(element["R"])), float(np.mean(element["Z"]))


def plot_description(ax, active, passive, boundary, *, title, z_offset=0.0):
    """Plot a raw machine description without first building a tokamak object."""
    for name, circuit in active.items():
        colour = "tab:red" if name == "vertical_control" else "tab:blue"
        parts = active_parts(circuit)
        for part in parts:
            r, z = centroid(part)
            ax.scatter(r, z + z_offset, s=36, color=colour, zorder=4)
        # Label a compound series circuit beside its upper bundle.
        r_label, z_label = max((centroid(part) for part in parts), key=lambda p: p[1])
        ax.annotate(
            name,
            (r_label, z_label + z_offset),
            fontsize=6,
            xytext=(3, 2),
            textcoords="offset points",
        )

    for element in passive:
        r = np.r_[element["R"], element["R"][0]]
        z = np.r_[element["Z"], element["Z"][0]] + z_offset
        ax.fill(r, z, color="0.65", alpha=0.55, linewidth=0.5, edgecolor="0.25")

    points = np.asarray(
        [[item["R"], item["Z"] + z_offset] for item in boundary]
    )
    ax.plot(*np.vstack((points, points[0])).T, color="black", linewidth=1.2)
    ax.axhline(0.0, color="0.35", linewidth=0.8, linestyle=":")
    ax.set(title=title, xlabel="R [m]", ylabel="Z [m]", aspect="equal")
    ax.set_xlim(0.05, 1.95)
    ax.set_ylim(-1.5, 1.5)


active_source, passive_source, limiter_source, wall_source = (
    mastu_like_source_machine()
)

fig, ax = plt.subplots(figsize=(6.5, 7))
plot_description(
    ax,
    active_source,
    passive_source,
    limiter_source,
    title="Original approximate machine",
)
plt.show()


## 2. Safety check, then explicit preparation

"Pre-flight" here is not a lightweight alternative implementation. It is an
intentional call to the normal preparation function with its strict defaults.
The function deep-copies the source, infers reflected pairs, fits the common
source plane $Z_0$, and classifies active winding parity. If it finds an odd
circuit, it raises before returning any prepared machine. This gives the user a
named list of sources that strict even evolution would be unable to represent,
without modifying the original dictionaries.

For reflection $\mathcal{R}_Z:(R,Z)\mapsto(R,2Z_0-Z)$, active parity is

$$
\mathcal{R}_ZG_{\mathrm{coil}}=qG_{\mathrm{coil}},\qquad q\in\{+1,-1\}.
$$

Only $q=+1$ sources can remain independent coordinates in a strictly even
model. After inspecting the expected error, the second call explicitly sets
`exclude_odd_active=True`. That call reruns the complete pipeline, records the
excluded circuit, shifts all retained geometry by $-Z_0$, averages reflected
pairs, and returns the `prepared` object used below.


In [ ]:
# Demonstration only: use the strict default policy to show which circuits
# prevent strict even evolution. Normal user code can omit this try/except
# once the exclusion decision has been made.
# The expected exception is the result of this safety check, not a failed
# attempt to build the solver-facing machine.
try:
    prepare_up_down_symmetric_machine(
        active_source,
        passive_source,
        limiter_data=limiter_source,
        wall_data=wall_source,
        z_midplane="auto",
    )
except ValueError as error:
    print("Safety check stopped preparation:")
    print(" ", error)

# Functional call: permit omission of the named odd source, then run
# the full recentering and geometry-averaging pipeline.
prepared = prepare_up_down_symmetric_machine(
    active_source,
    passive_source,
    limiter_data=limiter_source,
    wall_data=wall_source,
    z_midplane="auto",
    exclude_odd_active=True,
)

print(f"Fitted source midplane:      {1e3 * prepared.source_z_midplane:+.3f} mm")
print(f"Applied whole-machine shift: {1e3 * prepared.z_shift:+.3f} mm")
print(f"Midplane fit RMS:            {1e3 * prepared.midplane_fit_rms:.3f} mm")
print(f"Matched fit samples:         {prepared.midplane_fit_samples}")
print(f"Excluded odd circuits:       {prepared.excluded_odd_active_names}")
print(f"Active reflected pairs:      {len(prepared.active_pairs)}")
print(f"Passive reflected pairs:     {len(prepared.passive_pairs)}")


In [ ]:
# Show the inferred pairings on the unmodified source description. The
# connecting lines are diagnostic only; they are not conductor geometry.
fig, ax = plt.subplots(figsize=(6.5, 7))
plot_description(
    ax,
    active_source,
    passive_source,
    limiter_source,
    title="Inferred reflected pairs",
)
ax.axhline(
    prepared.source_z_midplane,
    color="tab:green",
    linewidth=1.3,
    label=f"fitted source midplane ({1e3 * prepared.source_z_midplane:.1f} mm)",
)

for pair in prepared.active_pairs:
    ru, zu = centroid(active_source[pair.upper_name])
    rl, zl = centroid(active_source[pair.lower_name])
    ax.plot([ru, rl], [zu, zl], color="tab:blue", alpha=0.35, linewidth=0.8)

passive_lookup = {item["name"]: item for item in passive_source}
for pair in prepared.passive_pairs:
    ru, zu = centroid(passive_lookup[pair.upper_name])
    rl, zl = centroid(passive_lookup[pair.lower_name])
    ax.plot([ru, rl], [zu, zl], color="tab:orange", alpha=0.35, linewidth=0.6)

ax.legend(loc="lower right")
plt.show()


## 3. Inspect the geometric change before accepting it

For each accepted pair, the reported mismatch is the RMS **point distance**
between the upper geometry and the reflected lower geometry before averaging:

$$
d_{\mathrm{RMS}}=
\sqrt{\frac{1}{N}\sum_{k=1}^{N}
\left\lVert\mathbf{x}_{u,k}-\mathcal{R}_Z\mathbf{x}_{l,k}\right\rVert^2}.
$$

This is supporting information. The magnetic fingerprint test in the next step
is normally the more important acceptance criterion because it measures the
field representation actually used by the solver.


In [ ]:
largest = prepared.largest_geometry_discrepancies(count=10)
print("Largest pre-symmetrisation geometry discrepancies")
for record in largest:
    print(
        f"{record.component:8s} {record.upper_name:18s} / "
        f"{record.lower_name:18s}: {1e3 * record.reflected_rms:6.3f} mm"
    )

# This guard evaluates the pre-averaging discrepancies stored in prepared.
# It does not perform another averaging pass or tune the geometry.
# A project can make this acceptance threshold stricter or looser.
GEOMETRY_ACCEPTANCE = 0.010  # 10 mm RMS point distance
prepared.check_geometry_tolerance(GEOMETRY_ACCEPTANCE)

fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharex=True, sharey=True)
plot_description(
    axes[0],
    active_source,
    passive_source,
    limiter_source,
    title="Source geometry",
)
plot_description(
    axes[1],
    prepared.active_coils_data,
    prepared.passive_coils_data,
    prepared.limiter_data,
    title="Exactly symmetric geometry",
)
plt.tight_layout()
plt.show()


## 4. Compare magnetic fingerprints

Let $G$ denote the sampled Green-function basis mapping unit conductor currents
to poloidal flux on the equilibrium grid. The aggregate change is

$$
\epsilon_G =
100\,\frac{\lVert G_{\mathrm{sym}}-G_{\mathrm{source}}\rVert_F}
{\lVert G_{\mathrm{source}}\rVert_F}.
$$

This comparison includes the fitted whole-machine shift and geometric
averaging. It is reported separately for active and passive conductors. A small
percentage means the symmetrisation has preserved the machine's magnetic
fingerprint closely; acceptance remains a modelling decision for the user.


In [ ]:
# Exclude the odd circuit from the source comparison so both bases have
# identical labels, ordering, and dimensions. The comparison then isolates
# recentering and geometric averaging, rather than circuit removal.
retained_source_active = {
    name: deepcopy(active_source[name]) for name in prepared.original_active_names
}

raw_tokamak = build_machine.tokamak(
    active_coils_data=retained_source_active,
    passive_coils_data=passive_source,
    limiter_data=limiter_source,
    wall_data=wall_source,
)
symmetric_tokamak = build_machine.tokamak(
    active_coils_data=prepared.active_coils_data,
    passive_coils_data=prepared.passive_coils_data,
    limiter_data=prepared.limiter_data,
    wall_data=prepared.wall_data,
)

eq_kwargs = dict(
    Rmin=0.05,
    Rmax=1.95,
    Zmin=-1.5,
    Zmax=1.5,
    nx=65,
    ny=97,
    psi=None,
)
# Equilibrium supplies a common grid and the sampled conductor Green basis;
# this audit does not require a Grad-Shafranov equilibrium solve.
raw_eq = equilibrium_update.Equilibrium(tokamak=raw_tokamak, **eq_kwargs)
symmetric_eq = equilibrium_update.Equilibrium(tokamak=symmetric_tokamak, **eq_kwargs)

# _vgreen is the internal conductor-to-grid poloidal-flux basis. Active rows
# precede passive rows; use the retained
# active count to split both machines at the same coordinate boundary.
n_active = len(prepared.original_active_names)
raw_active_g = raw_eq._vgreen[:n_active]
sym_active_g = symmetric_eq._vgreen[:n_active]
raw_passive_g = raw_eq._vgreen[n_active:]
sym_passive_g = symmetric_eq._vgreen[n_active:]


def percentage_change(original, updated):
    return 100 * np.linalg.norm(updated - original) / np.linalg.norm(original)


active_change = percentage_change(raw_active_g, sym_active_g)
passive_change = percentage_change(raw_passive_g, sym_passive_g)
print(f"Active Green-basis change:  {active_change:.4f}%")
print(f"Passive Green-basis change: {passive_change:.4f}%")

# The scalar percentages summarize the whole basis. Summing squared source
# rows gives a current-coordinate-independent basis magnitude at each grid
# point. Errors are normalized by its global maximum to avoid unstable local
# ratios where the source field basis is nearly zero.
active_strength = np.sqrt(np.sum(raw_active_g**2, axis=0))
passive_strength = np.sqrt(np.sum(raw_passive_g**2, axis=0))
active_error = np.sqrt(np.sum((sym_active_g - raw_active_g) ** 2, axis=0))
passive_error = np.sqrt(np.sum((sym_passive_g - raw_passive_g) ** 2, axis=0))
extent = [raw_eq.Rmin, raw_eq.Rmax, raw_eq.Zmin, raw_eq.Zmax]

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
maps = [
    (active_strength, "source active basis magnitude"),
    (100 * active_error / np.max(active_strength), "active change / max source [%]"),
    (passive_strength, "source passive basis magnitude"),
    (
        100 * passive_error / np.max(passive_strength),
        "passive change / max source [%]",
    ),
]
for ax, (field, title) in zip(axes.flat, maps):
    image = ax.imshow(field.T, origin="lower", extent=extent, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("R [m]")
    ax.set_ylabel("Z [m]")
    fig.colorbar(image, ax=ax, shrink=0.82)
plt.tight_layout()
plt.show()


## 5. Transform reconstructed currents

The prepared object preserves the relationship between the original independent
active circuits and the reduced even circuits. In matrix form,

$$
\mathbf{I}_+ = T_+\mathbf{I},\qquad
\mathbf{I}_- = T_-\mathbf{I},
$$

and

$$
\mathbf{I}=U_+\mathbf{I}_+ + U_-\mathbf{I}_-.
$$

Therefore currents reconstructed on the original upper/lower circuits can be
projected onto the symmetric model without losing their provenance. Dropping
$\mathbf{I}_-$ gives the closest exactly even current vector in this basis.


In [ ]:
# A synthetic reconstructed current vector in the documented retained order.
# The excluded vertical_control circuit is intentionally not part of this
# transform because strict even evolution has no coordinate for it.
original_currents = np.array(
    [7_000.0]
    + [
        value
        for pair in [
            (2_100, 1_900),
            (-3_200, -2_800),
            (4_200, 3_900),
            (-2_500, -2_650),
            (3_100, 2_850),
            (-1_700, -1_450),
        ]
        for value in pair
    ]
)
assert len(original_currents) == len(prepared.original_active_names)

# Pair averages are the currents retained by strict even evolution; pair
# differences quantify the odd content discarded by that projection.
even_currents, odd_currents = prepared.split_active_currents(original_currents)
# Keeping both coordinates must reconstruct the source exactly. Omitting
# odd_currents deliberately returns the closest exactly even vector.
reconstructed = prepared.combine_active_currents(even_currents, odd_currents)
even_projection = prepared.combine_active_currents(even_currents)
np.testing.assert_allclose(reconstructed, original_currents)

print("Original active order:", prepared.original_active_names)
print("Reduced even order:   ", prepared.even_active_names)
print("Odd pair coordinates: ", prepared.active_odd_names)
print(
    f"Reconstruction error:  "
    f"{np.linalg.norm(reconstructed - original_currents):.3e} A"
)

x = np.arange(len(original_currents))
width = 0.38
fig, axes = plt.subplots(2, 1, figsize=(11, 7), constrained_layout=True)
axes[0].bar(x - width / 2, original_currents, width, label="original")
axes[0].bar(x + width / 2, even_projection, width, label="even projection")
axes[0].set_xticks(x, prepared.original_active_names, rotation=55, ha="right")
axes[0].set_ylabel("current [A]")
axes[0].set_title("Original currents and their exactly even projection")
axes[0].legend()

odd_x = np.arange(len(odd_currents))
axes[1].bar(odd_x, odd_currents, color="tab:orange")
axes[1].set_xticks(odd_x, prepared.active_odd_names, rotation=35, ha="right")
axes[1].set_ylabel("odd current coordinate [A]")
axes[1].set_title("Discarded upper/lower current differences")
plt.show()


## 6. Build the machine used by an even solver

`even_active_coils_data` connects each retained upper/lower pair as one series
circuit. Passive elements retain their labels because their even and odd normal
modes are selected later by the evolutive solver.

The passive reflection matrix $S_p$ records every parent pairing. It is a
symmetric involution,

$$
S_p^T=S_p,\qquad S_p^2=I,
$$

so the passive-current even projection is
$\mathbf{I}_{p,+}=(I+S_p)\mathbf{I}_p/2$.


In [ ]:
# Build the representation consumed by a strict even solver. Active
# reflected pairs are series-connected into one current coordinate.
even_tokamak = build_machine.tokamak(
    active_coils_data=prepared.even_active_coils_data,
    passive_coils_data=prepared.passive_coils_data,
    limiter_data=prepared.limiter_data,
    wall_data=prepared.wall_data,
)

# Passive elements remain separate in the machine description. Their
# reflection operator lets the evolutive solver select even passive modes.
passive_reflection = prepared.passive_reflection_operator
np.testing.assert_allclose(passive_reflection, passive_reflection.T)
np.testing.assert_allclose(
    passive_reflection @ passive_reflection,
    np.eye(len(passive_reflection)),
)

fig, ax = plt.subplots(figsize=(6.5, 7))
plot_description(
    ax,
    prepared.even_active_coils_data,
    prepared.passive_coils_data,
    prepared.limiter_data,
    title="Final machine for strict even evolution",
)
plt.show()

# (I + S_p) / 2 projects passive currents onto the even subspace. The trace
# of a projector is its rank, hence the number of retained passive modes.
n_passive_even = int(
    np.trace((np.eye(len(passive_reflection)) + passive_reflection) / 2)
)
print("Final active series circuits:", prepared.even_active_names)
print("Final passive elements:      ", len(prepared.passive_names))
print("Passive even coordinates:    ", n_passive_even)
print("Odd source circuits omitted: ", prepared.excluded_odd_active_names)


## What the preparation does, and what it does not do

The preparation:

- fits and removes a common vertical offset when requested;
- identifies reflected element pairs;
- records pre-averaging geometric changes for inspection;
- averages candidate geometry onto exact reflection symmetry;
- detects and explicitly records electrically odd active circuits;
- preserves transforms between original and even/odd current coordinates;
- returns the passive reflection operator needed by reduced dynamics.

The magnetic-fingerprint comparison is performed separately by this notebook;
it is not part of `prepare_up_down_symmetric_machine()`.

It does **not** decide whether the approximation is physically acceptable.
Geometry and fingerprint thresholds remain user choices. It also does not model
an excluded odd circuit: that requires a separate prescribed-odd or
vertical-lifting model rather than the strict even solver.
